# Skin Lesion Bias Reduction — Colab training

Runs the EfficientNetV2-B0 baseline classifier on a Colab GPU using the project code in `src/` and a Fitzpatrick17k dataset you've already uploaded.

**Order of operations**
1. Confirm GPU + mount Drive (if used)
2. Point the notebook at your code + data
3. Install dependencies
4. Train the baseline (with class weights, unfreeze schedule, save-best-by-val-loss)
5. Evaluate the best checkpoint and render a markdown bias report
6. *Optional* — kick off the cGAN training (placeholder, run later)

## 1. Verify GPU and Colab environment

In [1]:
import sys

IN_COLAB = "google.colab" in sys.modules
print("Colab:", IN_COLAB)

import torch
print("torch:", torch.__version__, "cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

Colab: True
torch: 2.10.0+cu128 cuda available: True
GPU: NVIDIA A100-SXM4-80GB
name, memory.total [MiB], memory.free [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB, 81148 MiB


## 2. Connect your code and data

Two common layouts work:

- **Drive layout** — repo and dataset both sit under `MyDrive`. Mount Drive and point `PROJECT_ROOT` at the repo there. Outputs persist between sessions.
- **Local Colab layout** — clone the repo into `/content/` and put the dataset under `/content/dataset/`. Faster I/O, but everything is wiped when the runtime ends.

Edit `PROJECT_ROOT`, `DATASET_CSV`, and `IMAGE_DIR` in the cell below to match where you uploaded things.

In [26]:
# Mount Drive only if you're using the Drive layout. Skip this cell otherwise.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [34]:
from pathlib import Path

# === EDIT THESE ===
PROJECT_ROOT = Path("/content/drive/MyDrive/SkinLesionBiasReduction")  # repo root containing src/, dataset/, run_*.sh
IMAGE_DIR    = PROJECT_ROOT / "dataset/images"
# ==================

print("PROJECT_ROOT:", PROJECT_ROOT)
print("IMAGE_DIR:   ", IMAGE_DIR,   "exists:", IMAGE_DIR.exists())
assert PROJECT_ROOT.exists(), f"PROJECT_ROOT does not exist: {PROJECT_ROOT}"
assert IMAGE_DIR.exists(),    f"IMAGE_DIR does not exist:    {IMAGE_DIR}"

%cd $PROJECT_ROOT

PROJECT_ROOT: /content/drive/MyDrive/SkinLesionBiasReduction
IMAGE_DIR:    /content/drive/MyDrive/SkinLesionBiasReduction/dataset/images exists: True
/content/drive/MyDrive/SkinLesionBiasReduction


In [ ]:
# Alternative: if you don't have the repo on Drive, clone it into /content/ and copy your dataset in.
# Uncomment, replace the URL with your fork, and re-run cell `configure-paths` with PROJECT_ROOT=/content/SkinLesionBiasReduction.
# !git clone git@github.com:hoangnam310/SkinLesionBiasReduction.git

In [36]:
!ls dataset

fitzpatrick17k_cleaned.gsheet  fitzpatrick17k.gsheet  images


## 3. Install dependencies

Colab images already include `torch`, `torchvision`, `numpy`, `pandas`, `Pillow`, `tqdm`, `matplotlib`, and `scipy`. The trainer additionally needs **timm** and **scikit-learn** (sklearn is usually preinstalled, timm usually is not). Tensorboard is optional and is also usually preinstalled.

In [8]:
!pip install --quiet timm 'scikit-learn>=1.3'

## 4. Train the baseline classifier

We start at **64×64** for fast iteration. Once the recipe looks healthy, bump `IMAGE_SIZE` to 224 and re-run.

Key flags being used:
- `--class_weights` — inverse-frequency weighted CrossEntropyLoss; the single biggest fix from the prior run's bias analysis.
- `--freeze_backbone --unfreeze_epoch 3 --fine_tune_lr 1e-5` — three warmup epochs on the head, then full fine-tune.
- The trainer also auto-saves the best checkpoint by val_loss across all epochs (no extra flag needed).

In [38]:
IMAGE_SIZE   = 64
EPOCHS       = 1
BATCH_SIZE   = 128       # T4 / A100 handle this comfortably at 64x64. Drop to 32 at 224x224.
LR           = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS  = 4

OUTPUT_DIR = PROJECT_ROOT / "outputs/baseline_efficientnet"
print("Outputs will land under:", OUTPUT_DIR)

Outputs will land under: /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet


In [ ]:
!python src/train_baseline_efficientnet.py \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{IMAGE_DIR}" \
    --image_size {IMAGE_SIZE} \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --weight_decay {WEIGHT_DECAY} \
    --num_workers {NUM_WORKERS} \
    --class_weights \
    --freeze_backbone --unfreeze_epoch 3 --fine_tune_lr 1e-5 \
    --output_dir "{OUTPUT_DIR}" \
    --device cuda

Using device: cuda
Image size: 64x64
Train batches: 100, Val batches: 25
Pretrained: True
Freeze backbone: True
model.safetensors: 100% 28.8M/28.8M [00:01<00:00, 17.7MB/s]
Training classifier head only (backbone frozen).
Class weights (benign, malignant, non-neoplastic): [2.4567, 2.4956, 0.4562]


## 5. Evaluate the best checkpoint

`train_baseline_efficientnet.py` restores best-by-val-loss weights before the final eval, so the auto-generated `metrics.json` already uses that snapshot. We additionally re-run `evaluate.py` to write `logs/evaluation_metrics.json` (with bias breakdowns) and then render a markdown report.

In [ ]:
runs = sorted(OUTPUT_DIR.glob("*/checkpoint.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
assert runs, f"No checkpoint.pt under {OUTPUT_DIR}"
LATEST_CKPT = runs[0]
print("Latest checkpoint:", LATEST_CKPT)

In [ ]:
!python src/evaluate.py \
    --checkpoint "{LATEST_CKPT}" \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{IMAGE_DIR}" \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS} \
    --device cuda \
    --logs_dir "{PROJECT_ROOT / 'logs'}"

In [ ]:
!python src/metrics_report.py \
    --json "{PROJECT_ROOT / 'logs' / 'evaluation_metrics.json'}"

In [ ]:
from IPython.display import Markdown, display

run_name = LATEST_CKPT.parent.name
report_path = PROJECT_ROOT / "logs" / f"{run_name}_report.md"
print("Report:", report_path)
display(Markdown(report_path.read_text()))

## 6. Scale up the image size (when ready)

Once the 64×64 numbers look directionally right (macro AUROC clearly above 0.63, malignant AUROC by FST less skewed), re-run section **4** with these overrides and let section **5** evaluate the new run:

```python
IMAGE_SIZE = 224
EPOCHS     = 40
BATCH_SIZE = 32   # may need 16 on a T4 to fit memory
```

Everything else stays the same.

## 7. Optional — train the cGAN later

The generative side lives in `src/train.py` (vanilla cGAN; the WGAN-GP critic exists in `src/cgan.py` but the WGAN trainer hasn't been committed yet). To run on Colab once you're ready:

In [ ]:
# Uncomment when you want to start cGAN training. Outputs land under outputs/<timestamp>/.
# !python src/train.py \
#     --csv_path "{DATASET_CSV}" \
#     --image_dir "{IMAGE_DIR}" \
#     --epochs 200 \
#     --batch_size 64 \
#     --lr 0.0002 \
#     --output_dir "{PROJECT_ROOT / 'outputs'}" \
#     --device cuda

In [ ]:
# After (or during) cGAN training, watch losses + sample grids in TensorBoard:
# %load_ext tensorboard
# %tensorboard --logdir $PROJECT_ROOT/outputs